# DICE ITC 04: Case Study, Attribution, and LLM Triage

This notebook groups the most reviewer-facing interpretability material: the time-series overlay, tier/mechanism attribution, grounded LLM triage support, and residual evidence concentration.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
CFG_LABEL = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}


def _cfg_labels(values: pd.Series) -> list[str]:
    return [CFG_LABEL.get(str(v), str(v)) for v in values]


def render_tier_correlation_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    tier_case = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    tier_final = tier_case[tier_case["config"] == "tier0_tier1_tier2"].copy()
    tier_cols = ["tier0_share", "tier1_alt_share", "tier2_share"]

    tier_corr = tier_final[tier_cols].corr().round(4)
    tier_corr.to_csv(paper_full / "tier_share_correlation.csv")

    stressor_tier = tier_final.groupby("stressor", sort=False)[tier_cols].mean().reset_index()
    stressor_tier.to_csv(paper_full / "stressor_tier_share_summary.csv", index=False)

    tier_final["ternary_x"] = tier_final["tier1_alt_share"] + 0.5 * tier_final["tier2_share"]
    tier_final["ternary_y"] = (np.sqrt(3.0) / 2.0) * tier_final["tier2_share"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    im = axes[0].imshow(tier_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_xticks(range(3), ["Tier-0", "Tier-1", "Tier-2"], rotation=30, ha="right")
    axes[0].set_yticks(range(3), ["Tier-0", "Tier-1", "Tier-2"])
    axes[0].set_title("Tier-share correlation")
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f"{tier_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(stressor_tier))
    axes[1].bar(x, stressor_tier["tier0_share"], label="Tier-0")
    axes[1].bar(x, stressor_tier["tier1_alt_share"], bottom=stressor_tier["tier0_share"], label="Tier-1")
    axes[1].bar(
        x,
        stressor_tier["tier2_share"],
        bottom=stressor_tier["tier0_share"] + stressor_tier["tier1_alt_share"],
        label="Tier-2",
    )
    axes[1].set_xticks(x, stressor_tier["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Mean tier evidence by stressor")
    axes[1].legend(loc="upper right")

    triangle = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.5, np.sqrt(3.0) / 2.0],
            [0.0, 0.0],
        ]
    )
    axes[2].plot(triangle[:, 0], triangle[:, 1], color="black")
    for stressor, d in tier_final.groupby("stressor", sort=False):
        axes[2].scatter(d["ternary_x"], d["ternary_y"], s=36, alpha=0.8, label=stressor)
    axes[2].text(-0.04, -0.03, "Tier-0")
    axes[2].text(1.01, -0.03, "Tier-1")
    axes[2].text(0.46, np.sqrt(3.0) / 2.0 + 0.03, "Tier-2")
    axes[2].set_title("Per-case tier composition")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
    axes[2].legend(loc="upper right", fontsize=7)

    fig.tight_layout()
    png = paper_fig / "fig_tier_correlation_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_corr, stressor_tier, png


def render_bootstrap_confidence(
    case_pred: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
    samples: int = 1000,
    seed: int = 0,
) -> tuple[pd.DataFrame, Path]:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for cfg, d in case_pred.groupby("config", sort=False):
        stats: list[dict[str, float]] = []
        for _ in range(samples):
            sample = d.sample(n=len(d), replace=True, random_state=int(rng.integers(1 << 32)))
            if sample["label"].nunique() < 2:
                continue
            benign = sample[sample["label"] == 0]
            anomaly = sample[sample["label"] == 1]
            stats.append(
                {
                    "roc_auc_wc": roc_auc_score(sample["label"], sample["run_score_wc"]),
                    "pr_auc_wc": average_precision_score(sample["label"], sample["run_score_wc"]),
                    "benign_run_false_alarm_rate": benign["run_alert"].mean(),
                    "anomaly_run_detection_rate": anomaly["run_alert"].mean(),
                    "median_time_to_detect_s": anomaly.loc[
                        anomaly["run_alert"] == 1, "time_to_detect_s"
                    ].median(),
                }
            )

        boot = pd.DataFrame(stats)
        rows.append(
            {
                "config": cfg,
                "roc_auc_wc_lo": boot["roc_auc_wc"].quantile(0.025),
                "roc_auc_wc_hi": boot["roc_auc_wc"].quantile(0.975),
                "pr_auc_wc_lo": boot["pr_auc_wc"].quantile(0.025),
                "pr_auc_wc_hi": boot["pr_auc_wc"].quantile(0.975),
                "fpr_lo": boot["benign_run_false_alarm_rate"].quantile(0.025),
                "fpr_hi": boot["benign_run_false_alarm_rate"].quantile(0.975),
                "detect_lo": boot["anomaly_run_detection_rate"].quantile(0.025),
                "detect_hi": boot["anomaly_run_detection_rate"].quantile(0.975),
                "ttd_lo": boot["median_time_to_detect_s"].quantile(0.025),
                "ttd_hi": boot["median_time_to_detect_s"].quantile(0.975),
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(paper_full / "bootstrap_confidence_intervals.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(out))

    pr_mid = (out["pr_auc_wc_lo"] + out["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - out["pr_auc_wc_lo"], out["pr_auc_wc_hi"] - pr_mid])
    axes[0].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4)
    axes[0].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[0].set_title("Bootstrap AUC-PR CI")

    fpr_mid = (out["fpr_lo"] + out["fpr_hi"]) / 2.0
    fpr_err = np.vstack([fpr_mid - out["fpr_lo"], out["fpr_hi"] - fpr_mid])
    axes[1].errorbar(x, fpr_mid, yerr=fpr_err, fmt="o", capsize=4)
    axes[1].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[1].set_title("Bootstrap benign-FPR CI")

    ttd_mid = (out["ttd_lo"] + out["ttd_hi"]) / 2.0
    ttd_err = np.vstack([ttd_mid - out["ttd_lo"], out["ttd_hi"] - ttd_mid])
    axes[2].errorbar(x, ttd_mid, yerr=ttd_err, fmt="o", capsize=4)
    axes[2].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[2].set_title("Bootstrap time-to-detect CI")

    fig.tight_layout()
    png = paper_fig / "fig_bootstrap_confidence_intervals.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out, png


def export_llm_case_cards(
    out_full: Path,
    appendix_full: Path,
) -> pd.DataFrame:
    case_diag = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    cards = case_diag[case_diag["config"] == "tier0_tier1_tier2"].copy()
    keep_cols = [
        "case_id",
        "workload",
        "stressor",
        "dominant_tier",
        "dominant_mechanism",
        "tier0_share",
        "tier1_alt_share",
        "tier2_share",
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
        "top_feature_1",
        "top_feature_score_1",
        "top_feature_2",
        "top_feature_score_2",
        "top_feature_3",
        "top_feature_score_3",
        "top_feature_4",
        "top_feature_score_4",
        "top_feature_5",
        "top_feature_score_5",
        "top_mechanism_1",
        "top_mechanism_score_1",
        "top_mechanism_2",
        "top_mechanism_score_2",
        "top_mechanism_3",
        "top_mechanism_score_3",
    ]
    cards = cards[[c for c in keep_cols if c in cards.columns]].copy()

    def _case_card_json(row: pd.Series) -> str:
        payload = {
            "case_id": row.get("case_id"),
            "workload": row.get("workload"),
            "stressor": row.get("stressor"),
            "dominant_tier": row.get("dominant_tier"),
            "dominant_mechanism": row.get("dominant_mechanism"),
            "tier_share": {
                "tier0": row.get("tier0_share"),
                "tier1": row.get("tier1_alt_share"),
                "tier2": row.get("tier2_share"),
            },
            "mechanism_share": {
                "compute": row.get("compute_share"),
                "memory_io": row.get("memory_io_share"),
                "thermal_power": row.get("thermal_power_share"),
                "scheduler_runtime": row.get("scheduler_runtime_share"),
                "platform_pressure": row.get("platform_pressure_share"),
            },
            "top_features": [
                {"name": row.get(f"top_feature_{i}"), "score": row.get(f"top_feature_score_{i}")}
                for i in range(1, 6)
                if pd.notna(row.get(f"top_feature_{i}"))
            ],
            "top_mechanisms": [
                {"name": row.get(f"top_mechanism_{i}"), "score": row.get(f"top_mechanism_score_{i}")}
                for i in range(1, 4)
                if pd.notna(row.get(f"top_mechanism_{i}"))
            ],
        }
        return json.dumps(payload, sort_keys=True)

    cards["diagnostic_case_card_json"] = cards.apply(_case_card_json, axis=1)
    cards["reviewer_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "You are preparing a grounded DICE diagnostic note for a silicon-reliability reviewer. "
            "Use only the supplied case card. Do not invent missing evidence. "
            "Explain the dominant tier, dominant mechanism, and the top residual cues in plain English.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["triage_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Write a compact triage report with five fields: "
            "severity, likely subsystem, evidence summary, two follow-up measurements, and confidence. "
            "If the evidence is weak, say so directly.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["followup_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Recommend up to three next diagnostic steps. "
            "Each step must cite the specific feature or mechanism that motivated it.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards.to_csv(appendix_full / "llm_case_cards.csv", index=False)
    return cards


def export_llm_diagnostic_model_catalog(appendix_full: Path) -> pd.DataFrame:
    models = pd.DataFrame(
        [
            {
                "model_id": "Qwen/Qwen2.5-7B-Instruct",
                "deployment_role": "primary DICE baseline",
                "priority_rank": 1,
                "params_billions": 7.61,
                "context_tokens": 131072,
                "strengths": "Strong instruction following, structured output behavior, and long-context support.",
                "best_for_dice": "Primary grounded reviewer summaries and structured incident reports from exported case cards.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct",
            },
            {
                "model_id": "microsoft/Phi-4-mini-instruct",
                "deployment_role": "lightweight comparison",
                "priority_rank": 2,
                "params_billions": 3.8,
                "context_tokens": 128000,
                "strengths": "Small footprint, strong reasoning density, and good fit for constrained local diagnostics.",
                "best_for_dice": "Fast first-pass case summaries and follow-up recommendations on a laptop or edge workstation.",
                "source_url": "https://huggingface.co/microsoft/Phi-4-mini-instruct",
            },
            {
                "model_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
                "deployment_role": "ecosystem baseline",
                "priority_rank": 3,
                "params_billions": 8.0,
                "context_tokens": 128000,
                "strengths": "Broad tooling support, stable chat behavior, and strong general-purpose instruction tuning.",
                "best_for_dice": "Fallback baseline when the deployment stack already supports Llama-family models.",
                "source_url": "https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct",
            },
            {
                "model_id": "Qwen/Qwen2.5-14B-Instruct",
                "deployment_role": "stronger offline review",
                "priority_rank": 4,
                "params_billions": 14.7,
                "context_tokens": 131072,
                "strengths": "Higher-capacity structured reasoning while remaining practical for offline workstation use.",
                "best_for_dice": "Second-pass failure analysis and richer postmortem summaries after the detector has already raised a case.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-14B-Instruct",
            },
        ]
    ).sort_values('priority_rank').reset_index(drop=True)
    models.to_csv(appendix_full / "llm_diagnostic_model_catalog.csv", index=False)
    return models


def export_llm_diagnostic_prompt_bundle(
    cards: pd.DataFrame,
    models: pd.DataFrame,
    appendix_full: Path,
) -> pd.DataFrame:
    system_prompt = (
        "You are a DICE diagnostic copilot. You may use only the structured DICE evidence supplied to you. "
        "Do not claim access to raw telemetry, hidden logs, or external knowledge about the run. "
        "If the evidence is incomplete, say that the conclusion is tentative."
    )
    rows = []
    for _, model in models.iterrows():
        for _, card in cards.iterrows():
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "triage",
                    "system_prompt": system_prompt,
                    "user_prompt": card["triage_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "reviewer_summary",
                    "system_prompt": system_prompt,
                    "user_prompt": card["reviewer_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "followup",
                    "system_prompt": system_prompt,
                    "user_prompt": card["followup_prompt"],
                }
            )

    bundle = pd.DataFrame(rows)
    bundle.to_csv(appendix_full / "llm_diagnostic_prompt_bundle.csv", index=False)
    with (appendix_full / "llm_diagnostic_prompt_bundle.jsonl").open("w") as f:
        for row in bundle.to_dict(orient="records"):
            f.write(json.dumps(row) + "\n")
    return bundle



TIER_ALIAS_MAP = {
    "tier0": ["tier0", "tier-0", "tier 0", "tier-0 evidence", "tier 0 evidence"],
    "tier1_alt": ["tier1", "tier-1", "tier 1", "tier1_alt", "tier-1 evidence", "tier 1 evidence"],
    "tier2": ["tier2", "tier-2", "tier 2", "tier-2 evidence", "tier 2 evidence"],
}

MECHANISM_ALIAS_MAP = {
    "compute": ["compute"],
    "memory_io": ["memory_io", "memory/io", "memory io"],
    "thermal_power": ["thermal_power", "thermal/power", "thermal power"],
    "scheduler_runtime": ["scheduler_runtime", "scheduler runtime"],
    "platform_pressure": ["platform_pressure", "platform pressure"],
}


def _slugify_model_id(model_id: str) -> str:
    return model_id.replace('/', '__').replace('-', '_').replace('.', '_')


def _normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace('_', ' ')
    text = text.replace(':', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def _feature_aliases(name: str | float | None) -> set[str]:
    if pd.isna(name):
        return set()
    name = str(name)
    aliases = {name.lower(), _normalize_text(name)}
    if ':' in name:
        tail = name.split(':', 1)[1]
        aliases.add(tail.lower())
        aliases.add(_normalize_text(tail))
    return {a for a in aliases if a}


def _contains_any(text: str, aliases: set[str] | list[str]) -> bool:
    norm = _normalize_text(text)
    return any(alias and _normalize_text(alias) in norm for alias in aliases)


def run_transformers_llm_batch(
    prompt_bundle: pd.DataFrame,
    appendix_full: Path,
    model_id: str,
    prompt_type: str = "triage",
    max_cases: int = 8,
    max_new_tokens: int = 320,
    temperature: float = 0.0,
) -> pd.DataFrame:
    if importlib.util.find_spec("transformers") is None or importlib.util.find_spec("torch") is None:
        raise RuntimeError("transformers and torch must be installed to run local model inference.")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    subset = prompt_bundle[
        (prompt_bundle["model_id"] == model_id) & (prompt_bundle["prompt_type"] == prompt_type)
    ].copy().head(max_cases)
    if subset.empty:
        raise ValueError(f"No prompts found for model_id={model_id!r} and prompt_type={prompt_type!r}.")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )

    rows = []
    for row in subset.itertuples(index=False):
        messages = [
            {"role": "system", "content": row.system_prompt},
            {"role": "user", "content": row.user_prompt},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            text = row.system_prompt + "\n\n" + row.user_prompt

        model_inputs = tokenizer([text], return_tensors="pt")
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=max(temperature, 1e-5),
        )
        new_ids = generated_ids[:, model_inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
        rows.append(
            {
                "model_id": model_id,
                "case_id": row.case_id,
                "prompt_type": prompt_type,
                "response_text": response_text,
            }
        )

    outputs = pd.DataFrame(rows)
    slug = _slugify_model_id(model_id)
    outputs.to_csv(appendix_full / f"llm_outputs_{slug}_{prompt_type}.csv", index=False)
    with (appendix_full / f"llm_outputs_{slug}_{prompt_type}.jsonl").open("w") as f:
        for rec in outputs.to_dict(orient="records"):
            f.write(json.dumps(rec) + "\n")
    return outputs


def evaluate_llm_grounding_outputs(
    llm_outputs: pd.DataFrame,
    llm_cards: pd.DataFrame,
    appendix_full: Path,
    stem: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail = llm_outputs.merge(llm_cards, on="case_id", how="left", suffixes=("", "_card")).copy()

    supported_tier_threshold = 0.05
    supported_mech_threshold = 0.10
    global_mechanisms = list(MECHANISM_ALIAS_MAP.keys())
    global_tiers = list(TIER_ALIAS_MAP.keys())

    rows = []
    for row in detail.itertuples(index=False):
        text = str(row.response_text)
        dominant_tier = getattr(row, "dominant_tier")
        dominant_mechanism = getattr(row, "dominant_mechanism")

        tier_aliases = set(TIER_ALIAS_MAP.get(str(dominant_tier), []))
        dominant_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(dominant_mechanism), [str(dominant_mechanism)]))
        top_feature_aliases = _feature_aliases(getattr(row, "top_feature_1", None))
        top_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(getattr(row, "top_mechanism_1", '')), [str(getattr(row, "top_mechanism_1", ''))]))

        mentions_dominant_tier = _contains_any(text, tier_aliases)
        mentions_dominant_mechanism = _contains_any(text, dominant_mech_aliases)
        mentions_top_feature_1 = _contains_any(text, top_feature_aliases)
        mentions_top_mechanism_1 = _contains_any(text, top_mech_aliases)

        supported_tiers = {
            "tier0": getattr(row, "tier0_share", 0.0),
            "tier1_alt": getattr(row, "tier1_alt_share", 0.0),
            "tier2": getattr(row, "tier2_share", 0.0),
        }
        unsupported_tier_mentions = any(
            _contains_any(text, TIER_ALIAS_MAP[t]) and supported_tiers.get(t, 0.0) < supported_tier_threshold
            for t in global_tiers
        )

        supported_mechs = {
            "compute": getattr(row, "compute_share", 0.0),
            "memory_io": getattr(row, "memory_io_share", 0.0),
            "thermal_power": getattr(row, "thermal_power_share", 0.0),
            "scheduler_runtime": getattr(row, "scheduler_runtime_share", 0.0),
            "platform_pressure": getattr(row, "platform_pressure_share", 0.0),
        }
        unsupported_mechanism_mentions = any(
            _contains_any(text, MECHANISM_ALIAS_MAP[m]) and supported_mechs.get(m, 0.0) < supported_mech_threshold
            for m in global_mechanisms
        )

        cue_coverage = np.mean([
            float(mentions_dominant_tier),
            float(mentions_dominant_mechanism),
            float(mentions_top_feature_1),
            float(mentions_top_mechanism_1),
        ])

        rows.append(
            {
                "model_id": getattr(row, "model_id"),
                "case_id": getattr(row, "case_id"),
                "prompt_type": getattr(row, "prompt_type"),
                "mentions_dominant_tier": mentions_dominant_tier,
                "mentions_dominant_mechanism": mentions_dominant_mechanism,
                "mentions_top_feature_1": mentions_top_feature_1,
                "mentions_top_mechanism_1": mentions_top_mechanism_1,
                "grounded_core": bool(mentions_dominant_tier and mentions_dominant_mechanism),
                "cue_coverage": cue_coverage,
                "unsupported_tier_mentions": unsupported_tier_mentions,
                "unsupported_mechanism_mentions": unsupported_mechanism_mentions,
                "hallucination_flag": bool(unsupported_tier_mentions or unsupported_mechanism_mentions),
            }
        )

    scored = pd.DataFrame(rows)
    summary = (
        scored.groupby(["model_id", "prompt_type"], sort=False)
        .agg(
            n_outputs=("case_id", "count"),
            grounded_core_rate=("grounded_core", "mean"),
            dominant_tier_rate=("mentions_dominant_tier", "mean"),
            dominant_mechanism_rate=("mentions_dominant_mechanism", "mean"),
            top_feature_1_rate=("mentions_top_feature_1", "mean"),
            top_mechanism_1_rate=("mentions_top_mechanism_1", "mean"),
            mean_cue_coverage=("cue_coverage", "mean"),
            hallucination_rate=("hallucination_flag", "mean"),
        )
        .reset_index()
    )

    summary.to_csv(appendix_full / f"llm_grounding_summary_{stem}.csv", index=False)
    scored.to_csv(appendix_full / f"llm_grounding_detail_{stem}.csv", index=False)
    return summary, scored


def render_paper_performance_stack(
    case_pred: pd.DataFrame,
    overall_full: pd.DataFrame,
    sequential: pd.DataFrame,
    reliability: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    summary = (
        overall_full[
            [
                "config",
                "roc_auc",
                "pr_auc",
                "roc_auc_wc",
                "pr_auc_wc",
                "median_nominal_score_wc",
                "median_anomaly_score_wc",
            ]
        ]
        .merge(
            sequential[
                [
                    "config",
                    "anomaly_detect_rate",
                    "median_time_to_detect_s",
                ]
            ],
            on="config",
        )
        .merge(
            reliability[
                [
                    "config",
                    "target_alpha",
                    "benign_block_false_alarm_rate",
                ]
            ],
            on="config",
        )
    )
    summary["config_label"] = _cfg_labels(summary["config"])
    summary["reliability_margin"] = summary["target_alpha"] - summary["benign_block_false_alarm_rate"]
    summary.to_csv(paper_full / "paper_performance_stack_summary.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    final = case_pred[case_pred["config"] == "tier0_tier1_tier2"].copy()
    benign = final[final["label"] == 0]["run_score_wc"].to_numpy(dtype=float)
    anomaly = final[final["label"] == 1]["run_score_wc"].to_numpy(dtype=float)
    bp = axes[0, 0].boxplot([benign, anomaly], labels=["Benign", "Anomaly"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2563eb", "#dc2626"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[0, 0].set_title("A. Final-head score separation")
    axes[0, 0].set_ylabel("Workload-conditioned run score")

    base_color = "#94a3b8"
    wc_color = "#0f766e"
    for _, row in summary.iterrows():
        axes[0, 1].scatter(row["roc_auc"], row["pr_auc"], color=base_color, s=70)
        axes[0, 1].scatter(row["roc_auc_wc"], row["pr_auc_wc"], color=wc_color, s=90)
        axes[0, 1].annotate(
            row["config_label"],
            (row["roc_auc_wc"], row["pr_auc_wc"]),
            textcoords="offset points",
            xytext=(6, 6),
        )
        axes[0, 1].plot([row["roc_auc"], row["roc_auc_wc"]], [row["pr_auc"], row["pr_auc_wc"]], color="#475569")
    axes[0, 1].set_xlabel("Run-level ROC-AUC")
    axes[0, 1].set_ylabel("Run-level Average Precision")
    axes[0, 1].set_title("B. Digital-twin score refinement")

    x = np.arange(len(summary))
    axes[1, 0].bar(x, summary["benign_block_false_alarm_rate"], color="#f59e0b")
    axes[1, 0].axhline(float(summary["target_alpha"].iloc[0]), color="black", linestyle="--", linewidth=1.2)
    axes[1, 0].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 0].set_ylabel("Empirical benign block FAR")
    axes[1, 0].set_title("C. Calibrated reliability")

    bars = axes[1, 1].bar(x, summary["anomaly_detect_rate"], color="#16a34a", label="Detection rate")
    ax2 = axes[1, 1].twinx()
    ax2.plot(x, summary["median_time_to_detect_s"], color="#1d4ed8", marker="o", linewidth=2, label="Median TTD")
    axes[1, 1].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 1].set_ylabel("Run-level detection rate")
    ax2.set_ylabel("Median time-to-detect (s)")
    axes[1, 1].set_title("D. Operational decision performance")
    axes[1, 1].legend([bars], ["Detection rate"], loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_performance_stack.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return summary, png


def render_explainability_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Path]:
    tier_contrib = pd.read_csv(out_full / "stressor_tier_contributions.csv")
    mechanism = pd.read_csv(out_full / "mechanism_group_summary.csv")
    cm = pd.read_csv(out_full / "stressor_confusion_matrix.csv", index_col=0)

    tier_contrib.to_csv(paper_full / "paper_tier_contribution_summary.csv", index=False)
    mechanism.to_csv(paper_full / "paper_mechanism_summary.csv", index=False)
    cm.to_csv(paper_full / "paper_stressor_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

    x = np.arange(len(tier_contrib))
    axes[0].bar(x, tier_contrib["tier0_share"], label="Tier-0", color="#4E79A7")
    axes[0].bar(x, tier_contrib["tier1_alt_share"], bottom=tier_contrib["tier0_share"], label="Tier-1", color="#59A14F")
    axes[0].bar(
        x,
        tier_contrib["tier2_share"],
        bottom=tier_contrib["tier0_share"] + tier_contrib["tier1_alt_share"],
        label="Tier-2",
        color="#F28E2B",
    )
    axes[0].set_xticks(x, tier_contrib["stressor"], rotation=30, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("A. Tier contribution by stressor")
    axes[0].legend(loc="upper right")
    axes[0].grid(axis="y", alpha=0.20)

    mech_cols = [
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
    ]
    mech_labels = {
        "compute_share": "Compute",
        "memory_io_share": "Memory/IO",
        "thermal_power_share": "Thermal/Power",
        "scheduler_runtime_share": "Scheduler/Runtime",
        "platform_pressure_share": "Platform Pressure",
    }
    plot_df = mechanism[["stressor", *mech_cols]].copy().rename(columns=mech_labels)
    long_df = plot_df.melt(id_vars="stressor", var_name="mechanism", value_name="share")

    x_order = [mech_labels[c] for c in mech_cols]
    y_order = list(plot_df["stressor"])
    x_map = {name: idx for idx, name in enumerate(x_order)}
    y_map = {name: idx for idx, name in enumerate(y_order)}

    sc = axes[1].scatter(
        long_df["mechanism"].map(x_map),
        long_df["stressor"].map(y_map),
        s=1250 * long_df["share"] + 40,
        c=long_df["share"],
        cmap="YlGnBu",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in long_df.iterrows():
        axes[1].text(
            x_map[row["mechanism"]],
            y_map[row["stressor"]],
            f'{row["share"]:.2f}',
            ha="center",
            va="center",
            fontsize=9,
        )
    axes[1].set_xticks(range(len(x_order)), x_order, rotation=20, ha="right")
    axes[1].set_yticks(range(len(y_order)), y_order)
    axes[1].set_title("B. Mechanism fingerprint by stressor")
    axes[1].grid(alpha=0.15)
    fig.colorbar(sc, ax=axes[1], fraction=0.046, pad=0.04, label="Mean mechanism share")

    im = axes[2].imshow(cm.values, cmap="Blues")
    axes[2].set_xticks(range(len(cm.columns)), list(cm.columns), rotation=30, ha="right")
    axes[2].set_yticks(range(len(cm.index)), list(cm.index))
    axes[2].set_title("C. Stressor diagnosis confusion")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    png = paper_fig / "fig_paper_explainability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_contrib, mechanism, cm, png
def render_portability_dashboard(
    frontier: pd.DataFrame,
    holdout: pd.DataFrame,
    bootstrap_ci: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    view = frontier.copy()
    view["config_label"] = _cfg_labels(view["config"])
    holdout_view = holdout.copy()
    holdout_view["config_label"] = _cfg_labels(holdout_view["config"])
    boot_view = bootstrap_ci.copy()
    boot_view["config_label"] = _cfg_labels(boot_view["config"])

    portability_summary = view[
        [
            "config",
            "config_label",
            "n_features",
            "portable_pr_auc",
            "holdout_worst_pr_auc",
            "reliability_margin",
            "joint_detection_diagnosis",
        ]
    ].copy()
    portability_summary.to_csv(paper_full / "paper_portability_summary.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    scatter = axes[0].scatter(
        view["n_features"],
        view["portable_pr_auc"],
        s=view["joint_detection_diagnosis"].fillna(0.0) * 1800 + 140,
        c=view["reliability_margin"],
        cmap="viridis",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in view.iterrows():
        axes[0].annotate(row["config_label"], (row["n_features"], row["portable_pr_auc"]), textcoords="offset points", xytext=(6, 6))
    axes[0].set_xlabel("Median active features")
    axes[0].set_ylabel("Portable AUC-PR")
    axes[0].set_title("A. Observability-portability frontier")
    fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(holdout_view))
    width = 0.35
    axes[1].bar(x - width / 2.0, holdout_view["mean_pr_auc"], width=width, label="Mean holdout PR")
    axes[1].bar(x + width / 2.0, holdout_view["worst_pr_auc"], width=width, label="Worst holdout PR")
    axes[1].set_xticks(x, holdout_view["config_label"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title("B. Holdout portability")
    axes[1].legend(loc="upper right")

    x = np.arange(len(boot_view))
    pr_mid = (boot_view["pr_auc_wc_lo"] + boot_view["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - boot_view["pr_auc_wc_lo"], boot_view["pr_auc_wc_hi"] - pr_mid])
    axes[2].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4, color="#1d4ed8", label="AP CI")
    det_mid = (boot_view["detect_lo"] + boot_view["detect_hi"]) / 2.0
    det_err = np.vstack([det_mid - boot_view["detect_lo"], boot_view["detect_hi"] - det_mid])
    axes[2].errorbar(x, det_mid, yerr=det_err, fmt="o", capsize=4, color="#16a34a", label="Detect-rate CI")
    axes[2].set_xticks(x, boot_view["config_label"], rotation=30, ha="right")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("C. Bootstrap uncertainty")
    axes[2].legend(loc="lower right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_portability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return portability_summary, png


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Quick Jump: Digital-Twin Case Study

If you only want the clearest post-detection results for the paper, start here after the notebook has been run once:
- `## 8. Case Study: True Time-Series Overlay`
- `## 9. Tier and Mechanism Attribution Dashboard`
- `## 10. LLM-Assisted Grounded Triage Results`


## 8. Case Study: True Time-Series Overlay

This section shows DICE as a genuine virtual-system comparison.
The anomaly case is plotted against its workload-matched benign reference using the exported block traces from the notebook-local engine.


In [ ]:
display(Markdown("### True time-series overlay: anomaly vs benign reference"))

paper_full = OUT_PAPER / "full"
paper_full.mkdir(parents=True, exist_ok=True)

case_pred = pd.read_csv(OUT_FULL / "case_predictions.csv")
trace_path = OUT_FULL / "case_block_traces.csv"

if not trace_path.exists():
    display(Markdown("`case_block_traces.csv` not found yet. Rerun **Run End-to-End** once after the patch cell."))
else:
    trace_df = pd.read_csv(trace_path)
    final_cfg = "tier0_tier1_tier2"

    detected = (
        case_pred[
            (case_pred["config"] == final_cfg)
            & (case_pred["label"] == 1)
            & (case_pred["run_alert"] == 1)
        ]
        .sort_values("run_score_wc", ascending=False)
    )

    if len(detected) == 0:
        display(Markdown("No detected anomaly case found for the Tier-0/1/2 head."))
    else:
        chosen = detected.iloc[0]
        chosen_case = chosen["case_id"]
        chosen_workload = chosen["workload"]

        nominal_case = case_pred[
            (case_pred["config"] == final_cfg)
            & (case_pred["workload"] == chosen_workload)
            & (case_pred["stressor"] == "NOMINAL")
        ].iloc[0]["case_id"]

        anom_trace = trace_df[(trace_df["config"] == final_cfg) & (trace_df["case_id"] == chosen_case)].copy()
        nom_trace = trace_df[(trace_df["config"] == final_cfg) & (trace_df["case_id"] == nominal_case)].copy()

        merged = anom_trace.merge(
            nom_trace[["block_idx", "score"]].rename(columns={"score": "nominal_score"}),
            on="block_idx",
            how="inner",
        ).copy()

        merged["delta_score"] = merged["score"] - merged["nominal_score"]
        merged["block_time_min"] = merged["block_end_s"] / 60.0

        first_persist = merged.loc[merged["persist_alert"] == 1, "block_time_min"]
        first_persist_time = float(first_persist.iloc[0]) if len(first_persist) else np.nan

        fig, axes = plt.subplots(2, 1, figsize=(12.8, 7.2), sharex=True)

        axes[0].plot(
            merged["block_time_min"],
            merged["score"],
            color="#4E79A7",
            linewidth=2.2,
            label=f"Anomaly case: {chosen_case}",
        )
        axes[0].plot(
            merged["block_time_min"],
            merged["nominal_score"],
            color="#59A14F",
            linewidth=2.0,
            linestyle="--",
            label=f"Benign reference: {nominal_case}",
        )
        axes[0].axhline(float(chosen["tau"]), color="#E15759", linestyle=":", linewidth=2, label="Conformal threshold")

        if np.isfinite(first_persist_time):
            axes[0].axvline(first_persist_time, color="black", linestyle="--", linewidth=1.5, label="First persistent alert")

        axes[0].set_ylabel("Block score")
        axes[0].set_title("Digital-twin residual score over time", fontweight="bold")
        axes[0].legend(frameon=False, loc="upper right")
        axes[0].grid(alpha=0.20)

        axes[1].fill_between(
            merged["block_time_min"],
            0,
            merged["delta_score"],
            color="#B07AA1",
            alpha=0.55,
            label="Foreground residual gap",
        )

        alert_times = merged.loc[merged["persist_alert"] == 1, "block_time_min"]
        alert_vals = merged.loc[merged["persist_alert"] == 1, "delta_score"]
        if len(alert_times):
            axes[1].scatter(alert_times, alert_vals, color="#C62828", s=22, label="Persistent alert blocks", zorder=3)

        axes[1].axhline(0.0, color="#64748B", linewidth=1)
        axes[1].set_xlabel("Time (minutes)")
        axes[1].set_ylabel("Anomaly - benign score")
        axes[1].set_title("Foreground divergence from virtual benign background", fontweight="bold")
        axes[1].legend(frameon=False, loc="upper right")
        axes[1].grid(alpha=0.20)

        fig.suptitle(
            f"True time-series overlay for {chosen_workload} / {chosen['stressor']}",
            fontsize=16,
            fontweight="bold",
            y=0.98,
        )

        out_png = paper_full / "fig_true_timeseries_overlay.png"
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.close(fig)

        display(Image(filename=str(out_png)))


## 9. Tier and Mechanism Attribution Dashboard

These plots explain where the digital twin's evidence comes from. The tier panel shows which observability level dominates. The mechanism panel now uses a bubble map so stressor-to-stressor differences are easier to compare than in a stacked bar chart. The confusion matrix shows how often the diagnosis head confuses one stressor with another.


In [ ]:
tier_corr, stressor_tier, tier_corr_png = render_tier_correlation_dashboard(OUT_FULL, PAPER_FULL, PAPER_FIG)
tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)

print('Tier attribution summary')
display(tier_summary.round(4))
print('Mechanism attribution summary')
display(mechanism_summary.round(4))
print('Diagnosis confusion summary')
display(cm_summary.round(4))
print('Tier-share correlation matrix')
display(tier_corr.round(4))
print('Mean tier evidence by stressor')
display(stressor_tier.round(4))

display(Image(filename=str(explainability_png)))
display(Image(filename=str(tier_corr_png)))


## 10. LLM-Assisted Grounded Triage Results

This section treats the LLM layer as a secondary results subsection built on top of structured DICE evidence. The detector itself is unchanged. DICE still produces the anomaly score, threshold, alert decision, dominant tier, dominant mechanism, and ranked residual cues. The LLM layer only converts those structured outputs into reviewer-facing summaries, triage notes, and follow-up recommendations.

The notebook therefore reports a grounded diagnostics pack rather than an LLM-based detector. It exports structured case cards, a local-model catalog, and prompt bundles for three downstream tasks: reviewer summary, triage note, and follow-up recommendation.

The recommended evaluation order is:
- `Qwen/Qwen2.5-7B-Instruct` as the primary DICE baseline,
- `microsoft/Phi-4-mini-instruct` as the lightweight comparison,
- `meta-llama/Meta-Llama-3.1-8B-Instruct` as an ecosystem baseline,
- `Qwen/Qwen2.5-14B-Instruct` for stronger offline review.

If no local inference runtime is installed, the notebook still exports the grounded artifacts needed for later model execution. That keeps the LLM layer aligned with the paper narrative while preserving the detector boundaries.


In [ ]:
llm_cards = export_llm_case_cards(OUT_FULL, APPENDIX_FULL)
llm_models = export_llm_diagnostic_model_catalog(APPENDIX_FULL)
llm_models = llm_models.sort_values('priority_rank').reset_index(drop=True)
llm_prompt_bundle = export_llm_diagnostic_prompt_bundle(llm_cards, llm_models, APPENDIX_FULL)

llm_summary = pd.DataFrame([
    {
        'n_case_cards': len(llm_cards),
        'n_model_profiles': len(llm_models),
        'n_prompt_rows': len(llm_prompt_bundle),
        'prompt_types': ', '.join(sorted(llm_prompt_bundle['prompt_type'].unique())),
    }
])
print('LLM-assisted grounded triage summary')
display(llm_summary)
print('Case-card preview')
display(llm_cards[['case_id', 'workload', 'stressor', 'dominant_tier', 'dominant_mechanism']].head(10))
print('Suggested local/offline model catalog (primary baseline first)')
display(llm_models)
print('Prompt bundle preview')
display(llm_prompt_bundle[['model_id', 'case_id', 'prompt_type']].head(12))

runtime_modules = {}
for mod in ['transformers', 'torch', 'vllm', 'llama_cpp']:
    runtime_modules[mod] = importlib.util.find_spec(mod) is not None
print('Local inference runtime available:', runtime_modules)
if not any(runtime_modules.values()):
    print('No LLM runtime is installed in this environment. The notebook exported grounded prompts and model metadata only; actual model outputs still require a local inference backend.')


### Run a Local Grounded LLM

This cell runs one local model over the exported DICE case cards. It is disabled by default because the released environment does not currently include an inference runtime. When you install a runtime, start with `Qwen/Qwen2.5-7B-Instruct` as the main baseline and `microsoft/Phi-4-mini-instruct` as the lightweight comparison.


In [ ]:
RUN_LOCAL_LLM = False
LOCAL_LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LOCAL_LLM_PROMPT_TYPE = "triage"
MAX_LLM_CASES = 8
MAX_NEW_TOKENS = 320
LLM_TEMPERATURE = 0.0

llm_outputs = None
if RUN_LOCAL_LLM:
    try:
        llm_outputs = run_transformers_llm_batch(
            llm_prompt_bundle,
            APPENDIX_FULL,
            model_id=LOCAL_LLM_MODEL_ID,
            prompt_type=LOCAL_LLM_PROMPT_TYPE,
            max_cases=MAX_LLM_CASES,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=LLM_TEMPERATURE,
        )
        print("Generated local LLM outputs")
        display(llm_outputs.head(5))
    except Exception as e:
        print(f"Local LLM run failed: {e}")
else:
    print("Set RUN_LOCAL_LLM = True after installing a local inference runtime.")


### Score Grounding, Cue Coverage, and Hallucination Rate

This cell scores the generated LLM outputs against structured DICE evidence. It reports three simple diagnostics: grounding of the dominant tier/mechanism, cue coverage of the top-ranked evidence, and a conservative hallucination heuristic based on unsupported tier or mechanism mentions.


In [ ]:
if llm_outputs is None:
    candidate_slug = _slugify_model_id(LOCAL_LLM_MODEL_ID)
    candidate_csv = APPENDIX_FULL / f"llm_outputs_{candidate_slug}_{LOCAL_LLM_PROMPT_TYPE}.csv"
    if candidate_csv.exists():
        llm_outputs = pd.read_csv(candidate_csv)
    else:
        llm_outputs = None

if llm_outputs is None:
    print("No local LLM outputs found yet. Run the previous cell first or place a saved output CSV in the appendix folder.")
else:
    eval_stem = _slugify_model_id(str(llm_outputs["model_id"].iloc[0])) + "_" + str(llm_outputs["prompt_type"].iloc[0])
    llm_grounding_summary, llm_grounding_detail = evaluate_llm_grounding_outputs(
        llm_outputs,
        llm_cards,
        APPENDIX_FULL,
        stem=eval_stem,
    )
    print("Grounded LLM evaluation summary")
    display(llm_grounding_summary.round(4))
    print("Grounded LLM evaluation detail")
    display(llm_grounding_detail.head(10))


## 11. Residual Evidence Concentration

This section shows how much anomaly evidence is captured by the top residual contributors.
That is a more DICE-specific interpretability view than simple feature-frequency counts.


In [ ]:
case_diag = pd.read_csv(OUT_FULL / 'case_diagnosis_summary.csv')
anom_diag = case_diag[case_diag['label'] == 1].copy()
mech_cols = [
    'compute_contrib',
    'memory_io_contrib',
    'thermal_power_contrib',
    'scheduler_runtime_contrib',
    'platform_pressure_contrib',
]
anom_diag['total_evidence'] = anom_diag[mech_cols].sum(axis=1).replace(0.0, np.nan)

for k in range(1, 6):
    cols = [f'top_feature_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'feature_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

for k in range(1, 4):
    cols = [f'top_mechanism_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'mechanism_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

coverage_cols = [
    'feature_top1_coverage',
    'feature_top2_coverage',
    'feature_top3_coverage',
    'feature_top4_coverage',
    'feature_top5_coverage',
    'mechanism_top1_coverage',
    'mechanism_top2_coverage',
    'mechanism_top3_coverage',
]
evidence_frontier = anom_diag.groupby('config', sort=False)[coverage_cols].mean().reset_index()
evidence_frontier['label'] = evidence_frontier['config'].map(CFG_LABEL).fillna(evidence_frontier['config'])
evidence_frontier.to_csv(PAPER_FULL / 'residual_evidence_concentration.csv', index=False)

stressor_evidence = (
    anom_diag[anom_diag['config'] == 'tier0_tier1_tier2']
    .groupby('stressor', sort=False)[
        [
            'feature_top1_coverage',
            'feature_top3_coverage',
            'feature_top5_coverage',
            'mechanism_top1_coverage',
            'mechanism_top2_coverage',
            'mechanism_top3_coverage',
        ]
    ]
    .mean()
    .reset_index()
)
stressor_evidence.to_csv(PAPER_FULL / 'stressor_residual_evidence_concentration.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
feature_k = [1, 2, 3, 4, 5]
mechanism_k = [1, 2, 3]
for _, row in evidence_frontier.iterrows():
    axes[0].plot(feature_k, [row[f'feature_top{k}_coverage'] for k in feature_k], marker='o', linewidth=2, label=row['label'])
    axes[1].plot(mechanism_k, [row[f'mechanism_top{k}_coverage'] for k in mechanism_k], marker='o', linewidth=2, label=row['label'])
axes[0].set_xlabel('Top-k residual features')
axes[0].set_ylabel('Mean anomaly evidence coverage')
axes[0].set_xticks(feature_k)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title('Residual evidence concentration by feature rank')
axes[1].set_xlabel('Top-k mechanism groups')
axes[1].set_ylabel('Mean anomaly evidence coverage')
axes[1].set_xticks(mechanism_k)
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title('Residual evidence concentration by mechanism rank')
axes[1].legend(loc='lower right')
fig.tight_layout()
coverage_png = PAPER_FIG / 'fig_residual_evidence_concentration.png'
fig.savefig(coverage_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Residual evidence concentration across digital-twin heads')
display(evidence_frontier)

print('Final-head stressor evidence concentration')
display(stressor_evidence)

display(Image(filename=str(coverage_png)))


## Quick Jump: Paper Figures

If you only want the reviewer-facing paper artifacts, start here once the result CSVs already exist:
- `## 13. Paper-Ready Figure Bundle`
- then review the claim-boundary and reproducibility sections below.
